# Notebook 4 — Pipeline ETL completo: NIST ASD → MySQL
## Proyecto: Óptica y Fotónica — Primer Parcial

**Estudiante:** Perez Criollo Andres David

Este notebook ejecuta **de punta a punta y sin ningún paso manual** el ciclo completo de
datos: descarga desde el NIST (con caché) → limpieza y transformación → construcción de la
forma relacional → carga en MySQL. Es autocontenido: puede correrse de arriba a abajo sin
depender de que otro notebook se haya ejecutado antes.

**Requisito previo — orden obligatorio:**

```
1. docker compose up -d
2. Ejecutar sql/schema.sql en DBeaver (o via mysql client)
   — crea la base optica_y_fotonica (si no existe) y las 9 tablas VACIAS
3. Ejecutar ESTE notebook — las puebla
```

Este notebook **no crea la base ni las tablas**. Si `optica_y_fotonica` o alguna tabla
no existen todavía, se detiene con un mensaje claro (Sección 2) en vez de intentar
crearlas: el DDL vive solo en `sql/schema.sql`.

**Reejecución:** el notebook es idempotente. Cada vez que corre, **vacía y vuelve a cargar**
las 9 tablas (Sección 7), así que puede ejecutarse tantas veces como se quiera sin errores de
clave duplicada.

**Sobre el nivel de detalle:** este es un notebook de *pipeline*, no de exploración didáctica.
Cada decisión de limpieza (L1–L14) se aplica con un comentario breve; su justificación
completa —computacional y física, con las cifras que la respaldan— ya está documentada en
`notebooks/03_limpieza_datos.ipynb` y en `docs/03_limpieza_datos.md`. El
diseño de las 9 tablas y este pipeline están documentados en `docs/04_pipeline_etl_mysql.md`.

---
# Sección 1 — Librerías y conexión a MySQL

Se usa **SQLAlchemy** (ya viene en la imagen `scipy-notebook`) con el driver **PyMySQL**
(se instala si falta). La conexión usa el **nombre del servicio Docker** (`mysql`), no
`localhost`, para demostrar que los contenedores se comunican por la red interna definida en
`docker-compose.yml`.

In [1]:
import importlib
import subprocess
import sys

REQUERIDAS = {"pandas": "pandas", "numpy": "numpy", "sqlalchemy": "sqlalchemy",
              "pymysql": "pymysql", "requests": "requests"}

faltantes = []
for modulo, paquete in REQUERIDAS.items():
    try:
        importlib.import_module(modulo)
    except ImportError:
        faltantes.append(paquete)

if faltantes:
    print("Instalando paquetes faltantes:", faltantes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
else:
    print("Todas las librerias necesarias ya estan disponibles.")

Instalando paquetes faltantes: ['pymysql']


In [2]:
import os
import re
import time
import hashlib
from datetime import datetime
from io import StringIO

import numpy as np
import pandas as pd
import requests
import sqlalchemy as sa
from sqlalchemy import text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Credenciales tomadas de las mismas variables de entorno que usa docker-compose.yml
# (ver .env). Host = nombre del servicio Docker, NO localhost.
# La base optica_y_fotonica la crea sql/schema.sql; este notebook solo se conecta a ella.
DB_USER = os.environ.get("MYSQL_USER", "root")
DB_PASS = os.environ.get("MYSQL_ROOT_PASSWORD", "root")
DB_HOST = "mysql"
DB_PORT = 3306
DB_NAME = os.environ.get("MYSQL_DATABASE", "optica_y_fotonica")

URL_SERVIDOR = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/"
engine_servidor = sa.create_engine(URL_SERVIDOR, pool_pre_ping=True)

with engine_servidor.connect() as conn:
    version = conn.execute(text("SELECT VERSION()")).scalar()
    bases = {row[0] for row in conn.execute(text("SHOW DATABASES"))}

if DB_NAME not in bases:
    raise RuntimeError(
        f"La base de datos '{DB_NAME}' no existe todavia.\n"
        "Antes de correr este notebook hay que ejecutar 'sql/schema.sql' en DBeaver "
        "(o con 'docker exec -i mysql_container mysql -uroot -proot < sql/schema.sql').\n"
        "Ese script CREA la base optica_y_fotonica y las 9 tablas; este notebook solo las puebla."
    )

URL_CONEXION = f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sa.create_engine(URL_CONEXION, pool_pre_ping=True)

with engine.connect() as conn:
    base_actual = conn.execute(text("SELECT DATABASE()")).scalar()

print(f"Conexion exitosa a MySQL {version}")
print(f"Base de datos activa: {base_actual}")
print(f"Fecha de ejecucion  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Conexion exitosa a MySQL 8.4.11
Base de datos activa: optica_y_fotonica
Fecha de ejecucion  : 2026-09-21 06:34:12


---
# Sección 2 — Verificación de que el esquema ya existe

Este notebook **no ejecuta DDL**. Antes de tocar nada, comprueba que existe la base
`optica_y_fotonica` y que las 9 tablas de `sql/schema.sql` ya están creadas (en DBeaver).
Si falta la base o alguna tabla, se detiene con un mensaje explícito en vez de intentar
crearlas — el orden schema→datos es un requisito del proyecto, no un detalle de
implementación.

In [3]:
TABLAS_ESPERADAS = [
    "elemento", "tipo_transicion", "exactitud", "referencia",
    "espectro", "nivel_energia", "linea_espectral",
    "linea_referencia", "nivel_referencia",
]

inspector = sa.inspect(engine)
tablas_existentes = set(inspector.get_table_names())

faltan = [t for t in TABLAS_ESPERADAS if t not in tablas_existentes]

print("Tablas esperadas segun sql/schema.sql:")
for t in TABLAS_ESPERADAS:
    marca = "OK " if t in tablas_existentes else "NO EXISTE"
    print(f"  [{marca}] {t}")

if faltan:
    raise RuntimeError(
        "El esquema de la base de datos no esta creado todavia.\n"
        f"Faltan las tablas: {faltan}\n\n"
        "Antes de correr este notebook hay que ejecutar 'sql/schema.sql' en DBeaver "
        "(o con 'docker exec -i mysql_container mysql -uroot -proot < sql/schema.sql').\n"
        "Ese script crea la base optica_y_fotonica y las tablas. Este notebook no las crea."
    )

print("\nEl esquema ya esta creado. Se puede continuar con la carga de datos.")

Tablas esperadas segun sql/schema.sql:
  [OK ] elemento
  [OK ] tipo_transicion
  [OK ] exactitud
  [OK ] referencia
  [OK ] espectro
  [OK ] nivel_energia
  [OK ] linea_espectral
  [OK ] linea_referencia
  [OK ] nivel_referencia

El esquema ya esta creado. Se puede continuar con la carga de datos.


In [4]:
# Deben estar vacias en la primera ejecucion (o con datos de una corrida anterior,
# que la Seccion 7 vaciara antes de recargar).
print("Filas actuales en cada tabla:")
with engine.connect() as conn:
    for t in TABLAS_ESPERADAS:
        n = conn.execute(text(f"SELECT COUNT(*) FROM {t}")).scalar()
        print(f"  {t:<20} {n:>8,} filas")

Filas actuales en cada tabla:
  elemento                    0 filas
  tipo_transicion             0 filas
  exactitud                   0 filas
  referencia                  0 filas
  espectro                    0 filas
  nivel_energia               0 filas
  linea_espectral             0 filas
  linea_referencia            0 filas
  nivel_referencia            0 filas


---
# Sección 3 — Extracción (descarga con caché)

Mismos parámetros y misma estrategia de caché validados en `02_exploracion_caracterizacion_datos.ipynb`:
si el CSV crudo ya existe en `datos_originales/`, se lee de disco; si no, se descarga del
NIST. El archivo crudo **nunca se modifica**.

In [5]:
DIR_ORIGINALES = "datos_originales"
DIR_NIVELES = os.path.join(DIR_ORIGINALES, "niveles")
DIR_PROCESADOS = "datos_procesados"
os.makedirs(DIR_ORIGINALES, exist_ok=True)
os.makedirs(DIR_NIVELES, exist_ok=True)
os.makedirs(DIR_PROCESADOS, exist_ok=True)

FORZAR_DESCARGA = False  # True para volver a pedir todo al NIST


def obtener_csv_nist(url, parametros, ruta_cache, timeout=180):
    if os.path.exists(ruta_cache) and not FORZAR_DESCARGA:
        with open(ruta_cache, "r", encoding="utf-8") as f:
            return f.read(), "cache"
    respuesta = requests.get(url, params=parametros, timeout=timeout)
    respuesta.raise_for_status()
    with open(ruta_cache, "w", encoding="utf-8") as f:
        f.write(respuesta.text)
    return respuesta.text, "red"


def parsear_csv_nist(texto, nombre_espectro=None):
    lineas = [ln for ln in texto.split("\n") if ln.strip() and not ln.startswith("---")]
    df = pd.read_csv(StringIO("\n".join(lineas)), sep=",", low_memory=False, on_bad_lines="warn")
    fantasmas = [c for c in df.columns if str(c).startswith("Unnamed:") and df[c].isna().all()]
    if fantasmas:
        df = df.drop(columns=fantasmas)
    if nombre_espectro is not None:
        df.insert(0, "espectro", nombre_espectro)
    return df


print("Ayudantes de extraccion definidos.")

Ayudantes de extraccion definidos.


In [6]:
URL_LINES = "https://physics.nist.gov/cgi-bin/ASD/lines1.pl"
ESPECTRA_STR = ("H I;He I;Li I;Li II;Be I;Be II;B I;B II;C I;C II;"
                "N I;N II;O I;O II;F I;F II;Ne I;Ne II")

PARAMS_LINES = {
    "spectra": ESPECTRA_STR, "low_w": "", "upp_w": "", "unit": 1, "format": 2,
    "line_out": 0, "en_unit": 0, "output": 0, "show_obs_wl": 1, "show_calc_wl": 1,
    "unc_out": 1, "order_out": 0, "show_av": 2, "A_out": 0, "f_out": "on",
    "loggf_out": "on", "intens_out": "on", "allowed_out": 1, "forbid_out": 1,
    "conf_out": "on", "term_out": "on", "enrg_out": "on", "J_out": "on",
    "g_out": "on", "ids_out": 1, "bibrefs": 1,
    "submit": "Retrieve Data", "de": 0, "plot_out": 0, "I_scale_type": 1,
    "page_size": 15, "tsb_value": 0, "min_str": "", "max_str": "",
    "max_low_enrg": "", "max_upp_enrg": "", "min_accur": "", "min_intens": "",
}

RUTA_LINES = os.path.join(DIR_ORIGINALES, "nist_lines_H_Ne_original.csv")
texto_lines, origen_lines = obtener_csv_nist(URL_LINES, PARAMS_LINES, RUTA_LINES)
print(f"Lineas espectrales: origen={origen_lines}  bytes={os.path.getsize(RUTA_LINES):,}")

Lineas espectrales: origen=cache  bytes=5,418,514


In [7]:
URL_LEVELS = "https://physics.nist.gov/cgi-bin/ASD/energy1.pl"
ESPECTRA_LISTA = ["H I", "He I", "Li I", "Li II", "Be I", "Be II", "B I", "B II",
                  "C I", "C II", "N I", "N II", "O I", "O II", "F I", "F II", "Ne I", "Ne II"]

PARAMS_LEVELS_BASE = {
    "units": 1, "format": 2, "output": 0, "multiplet_ordered": 0,
    "conf_out": "on", "term_out": "on", "level_out": "on", "unc_out": "on",
    "j_out": "on", "g_out": "on", "ids_out": "on", "lande_out": "on",
    "perc_out": "on", "biblio": "on",
    "submit": "Retrieve Data", "page_size": 15, "temp": "", "de": 0,
}

lista_dfs = []
origenes_niveles = []
for espectro in ESPECTRA_LISTA:
    parametros = dict(PARAMS_LEVELS_BASE)
    parametros["spectrum"] = espectro
    ruta = os.path.join(DIR_NIVELES, f"nist_levels_{espectro.replace(' ', '_')}_original.csv")
    texto, origen = obtener_csv_nist(URL_LEVELS, parametros, ruta, timeout=120)
    origenes_niveles.append(origen)
    df_esp = parsear_csv_nist(texto, nombre_espectro=espectro)
    lista_dfs.append(df_esp)
    if origen == "red":
        time.sleep(1)

df_levels_raw = pd.concat(lista_dfs, ignore_index=True)
RUTA_LEVELS = os.path.join(DIR_ORIGINALES, "nist_levels_todos_original.csv")
df_levels_raw.to_csv(RUTA_LEVELS, index=False, encoding="utf-8")

print(f"Niveles de energia: {len(df_levels_raw):,} filas de {len(ESPECTRA_LISTA)} espectros")
print(f"  Origen: {pd.Series(origenes_niveles).value_counts().to_dict()}")

Niveles de energia: 5,752 filas de 18 espectros
  Origen: {'cache': 18}


In [8]:
# Reconstruccion del medio (aire/vacio) desde la posicion en el archivo crudo de lineas.
# El NIST responde en 3 bloques y el del medio usa una convencion de medida distinta,
# sin marcarlo en ninguna columna. Ver docs/03_limpieza_datos.md, decision L2.
with open(RUTA_LINES, "r", encoding="utf-8") as f:
    texto_lines = f.read()

utiles = [ln for ln in texto_lines.split("\n") if ln.strip() and not ln.startswith("---")]
cortes = [i for i, ln in enumerate(utiles) if ln.startswith("element,sp_num")]

medio_por_bloque = {ini: ("aire" if "obs_wl_air" in utiles[ini] else "vacio") for ini in cortes}
medio_filas = []
for idx in range(1, len(utiles)):
    bloque = max(c for c in cortes if c <= idx)
    medio_filas.append(medio_por_bloque[bloque])

df_lines_raw = parsear_csv_nist(texto_lines)
assert len(df_lines_raw) == len(medio_filas)
df_lines_raw["medio"] = medio_filas

print(f"Lineas espectrales: {len(df_lines_raw):,} filas")
print(pd.Series(medio_filas).value_counts().to_string())

Lineas espectrales: 18,290 filas
aire     10429
vacio     7861


---
# Sección 4 — Transformación: limpieza (L1–L14)

Se aplican, de forma condensada, las 14 decisiones ya justificadas y verificadas en
`03_limpieza_datos.ipynb`. El principio que las gobierna: **en un dataset científico un valor
ausente es información** — no se imputa en variables críticas, no se elimina ningún atípico.

In [9]:
def decodificar_exportacion(df):
    # L3: quita el envoltorio ="valor" del CSV del NIST; "" -> NaN.
    salida = df.copy()
    for col in salida.columns:
        if pd.api.types.is_numeric_dtype(salida[col]):
            continue
        s = salida[col].astype(str).str.strip()
        s = s.str.replace(r'^="', "", regex=True).str.replace(r'"$', "", regex=True).str.strip()
        salida[col] = s.replace({"": np.nan, "nan": np.nan, "None": np.nan}).mask(salida[col].isna())
    return salida


df_lines = decodificar_exportacion(df_lines_raw)
df_levels = decodificar_exportacion(df_levels_raw)
print(f"L3 - Decodificado. Ausencias reales: lineas={int(df_lines.isna().sum().sum()):,}  "
      f"niveles={int(df_levels.isna().sum().sum()):,}")

L3 - Decodificado. Ausencias reales: lineas=90,149  niveles=23,013


In [10]:
# L1: eliminar las filas de cabecera incrustadas (unica decision que elimina filas).
antes = len(df_lines)
df_toclean = df_lines[df_lines["element"] != "element"].reset_index(drop=True).copy()
print(f"L1 - Cabeceras incrustadas eliminadas: {antes - len(df_toclean)}")

L1 - Cabeceras incrustadas eliminadas: 2


In [11]:
N_AIRE = 1.00028  # indice de refraccion del aire

def quitar_artefacto(serie):
    s = serie.astype(str).str.strip()
    s = s.str.replace(r'^=', "", regex=True).str.replace(r'^"|"$', "", regex=True).str.strip()
    return s.mask(serie.isna())


def extraer_energia(serie):
    # L4: [valor]=interpolada, valor+x=origen desconocido. Se extrae el numero
    # y se conserva el significado en dos banderas, en vez de perder el dato.
    s = serie.astype(str)
    interpolada = s.str.contains(r"\[", na=False).where(serie.notna())
    origen_desc = s.str.contains(r"\+[xyzXYZ]", na=False).where(serie.notna())
    limpio = s.str.replace(r"[\[\]]", "", regex=True).str.replace(r"\+[xyzXYZ]", "", regex=True).str.strip()
    return pd.to_numeric(limpio, errors="coerce"), interpolada, origen_desc


PATRON_J = re.compile(r"\d+(/\d+)?")


def parsear_J(v):
    # L5: convierte fracciones (3/2 -> 1.5); deja NaN en los no resueltos
    # (---, 0,1,2, 1/2 or 3/2, etc.), que son mas variados que solo '---'.
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if not PATRON_J.fullmatch(s):
        return np.nan
    if "/" in s:
        a, b = s.split("/")
        return float(a) / float(b)
    return float(s)


print("Ayudantes de limpieza definidos (L4, L5).")

Ayudantes de limpieza definidos (L4, L5).


In [12]:
for origen, destino in [("Ei(cm-1)", "energia_inferior_cm1"), ("Ek(cm-1)", "energia_superior_cm1")]:
    valor, interp, desc = extraer_energia(df_toclean[origen])
    df_toclean[destino] = valor
    df_toclean[destino + "_interpolada"] = interp
    df_toclean[destino + "_origen_desconocido"] = desc

s_ritz = df_toclean["ritz_wl_vac(nm)"].astype(str)
df_toclean["longitud_onda_ritz_nm"] = pd.to_numeric(
    s_ritz.str.replace(r"\+$", "", regex=True).str.strip(), errors="coerce")
df_toclean["ritz_es_limite"] = s_ritz.str.endswith("+", na=False).where(df_toclean["ritz_wl_vac(nm)"].notna())
df_toclean["longitud_onda_obs_nm"] = pd.to_numeric(df_toclean["obs_wl_vac(nm)"], errors="coerce")

s_int = df_toclean["intens"].astype(str)
df_toclean["intensidad_valor"] = pd.to_numeric(
    s_int.str.extract(r"([0-9]*\.?[0-9]+(?:[eE][+-]?[0-9]+)?)")[0], errors="coerce")
banderas = s_int.str.replace(r"[0-9.]", "", regex=True).str.strip()
df_toclean["intensidad_bandera"] = banderas.replace("", np.nan).where(df_toclean["intens"].notna())

print("L4 - Notacion del NIST interpretada (energias, Ritz, intensidad).")

L4 - Notacion del NIST interpretada (energias, Ritz, intensidad).


In [13]:
for origen, destino in [("J_i", "j_inferior"), ("J_k", "j_superior")]:
    df_toclean[destino + "_texto"] = df_toclean[origen]
    df_toclean[destino] = df_toclean[origen].map(parsear_J)
    df_toclean[destino + "_resuelto"] = df_toclean[destino].notna().where(df_toclean[origen].notna())

df_niv = df_levels.copy()
df_niv["j_texto"] = df_niv["J"]
df_niv["j_valor"] = df_niv["J"].map(parsear_J)
df_niv["j_resuelto"] = df_niv["j_valor"].notna().where(df_niv["J"].notna())
df_niv["g"] = pd.to_numeric(df_niv["g"], errors="coerce")

ev = df_niv["j_valor"].notna() & df_niv["g"].notna()
assert (df_niv.loc[ev, "g"] == 2 * df_niv.loc[ev, "j_valor"] + 1).all(), "Inconsistencia g=2J+1"
print(f"L5 - J parseado. Verificado g=2J+1 en {int(ev.sum()):,} niveles (100%).")

L5 - J parseado. Verificado g=2J+1 en 5,570 niveles (100%).


In [14]:
# L6: proteger identificadores como texto (evita perder ceros a la izquierda).
for col in ["ID_i", "ID_k"]:
    df_toclean[col] = df_toclean[col].astype("string")
df_niv["Level ID"] = df_niv["Level ID"].astype("string")

# L7: normalizar categoricas.
df_toclean.loc[df_toclean["Type"] == "2P", "Type"] = "UT"  # anomalo detectado en la exploracion
df_toclean["tipo_transicion"] = df_toclean["Type"].fillna("E1")
ESCALA_ACC = ["AAA", "AA", "A+", "A", "B+", "B", "B'", "C+", "C", "C'", "D+", "D", "E"]
df_toclean["exactitud"] = pd.Categorical(df_toclean["Acc"], categories=ESCALA_ACC, ordered=True)
for col in ["element", "conf_i", "term_i", "conf_k", "term_k"]:
    df_toclean[col] = df_toclean[col].astype("string").str.strip()
for col in ["Configuration", "Term"]:
    df_niv[col] = df_niv[col].astype("string").str.strip()

print("L6/L7 - Identificadores protegidos, categoricas normalizadas.")

L6/L7 - Identificadores protegidos, categoricas normalizadas.


In [15]:
# L8: longitud de onda con respaldo Ritz (decision de mayor impacto: casi triplica
# la base de evidencia sin imputar ningun valor).
df_toclean["longitud_onda_nm"] = df_toclean["longitud_onda_obs_nm"].fillna(df_toclean["longitud_onda_ritz_nm"])
df_toclean["longitud_onda_origen"] = np.where(df_toclean["longitud_onda_obs_nm"].notna(), "observada", "ritz")
df_toclean.loc[df_toclean["longitud_onda_nm"].isna(), "longitud_onda_origen"] = np.nan

# L9: Aki NO se imputa (es la variable dependiente de la pregunta cientifica).
df_toclean["aki_s1"] = pd.to_numeric(df_toclean["Aki(s^-1)"], errors="coerce")
df_toclean["fuerza_oscilador_fik"] = pd.to_numeric(df_toclean["fik"], errors="coerce")
df_toclean["log_gf"] = pd.to_numeric(df_toclean["log_gf"], errors="coerce")
df_toclean["incertidumbre_obs_nm"] = pd.to_numeric(df_toclean["unc_obs_wl"], errors="coerce")
df_toclean["incertidumbre_ritz_nm"] = pd.to_numeric(df_toclean["unc_ritz_wl"], errors="coerce")

ROMANOS = {1: "I", 2: "II"}
sp = pd.to_numeric(df_toclean["sp_num"], errors="coerce")
df_toclean["estado_ionizacion"] = sp.astype("Int8")
df_toclean["espectro"] = df_toclean["element"].astype(str) + " " + sp.map(lambda v: ROMANOS.get(int(v)) if pd.notna(v) else None)

base_evidencia = int((df_toclean["longitud_onda_nm"].notna() & df_toclean["aki_s1"].notna()).sum())
print(f"L8/L9 - Longitud de onda con respaldo Ritz; Aki sin imputar. "
      f"Base de evidencia: {base_evidencia:,} lineas.")

L8/L9 - Longitud de onda con respaldo Ritz; Aki sin imputar. Base de evidencia: 12,603 lineas.


In [16]:
# L12: marcar duplicados fisicos sin eliminarlos (son mediciones independientes).
clave_fisica = ["element", "sp_num", "longitud_onda_nm", "conf_i", "term_i", "J_i", "conf_k", "term_k", "J_k"]
df_toclean["medicion_repetida"] = df_toclean.duplicated(subset=clave_fisica, keep=False)

# L11: descartar columnas sin valor informativo en niveles (>=69% nulos, redundantes).
df_niv["energia_ev"] = pd.to_numeric(df_niv["Level (eV)"], errors="coerce")
df_niv["energia_interpolada"] = (df_niv["Prefix"].astype(str).str.strip() == "[")
df_niv["incertidumbre_ev"] = pd.to_numeric(df_niv["Uncertainty (eV)"], errors="coerce")
df_niv = df_niv.drop(columns=["Lande", "Prefix", "Suffix", "Leading percentages"])

print(f"L11/L12 - {int(df_toclean['medicion_repetida'].sum())} mediciones repetidas marcadas; "
      "4 columnas de niveles descartadas.")
print("L13 - Valores atipicos: se conservan (son transiciones prohibidas, fisica real).")

L11/L12 - 70 mediciones repetidas marcadas; 4 columnas de niveles descartadas.
L13 - Valores atipicos: se conservan (son transiciones prohibidas, fisica real).


In [17]:
# ---- Validacion automatica (igual que la seccion 17 de 03_limpieza_datos.ipynb) ----
# Si algo falla aqui, NO se continua hacia la base de datos.
assert len(df_toclean) == 18288, f"Se esperaban 18288 lineas, hay {len(df_toclean)}"
assert (df_toclean["longitud_onda_nm"].dropna() > 0).all(), "Longitud de onda <= 0"
assert (df_toclean["aki_s1"].dropna() > 0).all(), "Aki <= 0"
comparables = df_toclean["energia_superior_cm1"].notna() & df_toclean["energia_inferior_cm1"].notna()
assert (df_toclean.loc[comparables, "energia_superior_cm1"]
        > df_toclean.loc[comparables, "energia_inferior_cm1"]).all(), "Ek <= Ei"
assert base_evidencia == 12603, f"Se esperaban 12603 lineas utiles, hay {base_evidencia}"

print("VALIDACION OK: 18288 filas, 0 valores fisicamente imposibles, base de evidencia = 12603.")

VALIDACION OK: 18288 filas, 0 valores fisicamente imposibles, base de evidencia = 12603.


---
# Sección 5 — Transformación a forma relacional

A partir del dataset limpio se construyen los **9 DataFrames** que corresponden 1:1 a las
tablas de `sql/schema.sql`, siguiendo el diseño documentado en
`docs/04_pipeline_etl_mysql.md`.

In [18]:
# ---- elemento ----------------------------------------------------------------
NOMBRES = {"H": "Hidrogeno", "He": "Helio", "Li": "Litio", "Be": "Berilio", "B": "Boro",
           "C": "Carbono", "N": "Nitrogeno", "O": "Oxigeno", "F": "Fluor", "Ne": "Neon"}
NUM_ATOMICO = {"H": 1, "He": 2, "Li": 3, "Be": 4, "B": 5, "C": 6, "N": 7, "O": 8, "F": 9, "Ne": 10}

simbolos = sorted(df_toclean["element"].dropna().unique(), key=lambda s: NUM_ATOMICO[s])
t_elemento = pd.DataFrame({
    "id_elemento": range(1, len(simbolos) + 1),
    "simbolo": simbolos,
    "nombre": [NOMBRES[s] for s in simbolos],
    "numero_atomico": [NUM_ATOMICO[s] for s in simbolos],
})
map_elemento = dict(zip(t_elemento["simbolo"], t_elemento["id_elemento"]))
print(f"elemento         : {len(t_elemento)} filas")

elemento         : 10 filas


In [19]:
# ---- tipo_transicion -----------------------------------------------------------
TIPOS = {
    "E1": ("Dipolo electrico", True, "Transicion permitida, la mas comun y brillante"),
    "E2": ("Cuadrupolo electrico", False, "Transicion prohibida"),
    "M1": ("Dipolo magnetico", False, "Transicion prohibida"),
    "M2": ("Cuadrupolo magnetico", False, "Transicion prohibida"),
    "UT": ("Sin clasificar", False, "Tipo no determinado por el NIST"),
}
t_tipo_transicion = pd.DataFrame([
    {"codigo": c, "nombre": n, "es_permitida": p, "descripcion": d}
    for c, (n, p, d) in TIPOS.items()
])
print(f"tipo_transicion  : {len(t_tipo_transicion)} filas")

# ---- exactitud -------------------------------------------------------------
ESCALA_ACC = ["AAA", "AA", "A+", "A", "B+", "B", "B'", "C+", "C", "C'", "D+", "D", "E"]
TOLERANCIA = [0.3, 1, 2, 3, 7, 10, 10, 18, 25, 25, 40, 50, 50]  # % maximo, escala NIST
t_exactitud = pd.DataFrame({
    "codigo": ESCALA_ACC,
    "orden": range(1, len(ESCALA_ACC) + 1),
    "tolerancia_max_pct": TOLERANCIA,
    "descripcion": [f"Incertidumbre estimada hasta {t}%" for t in TOLERANCIA],
})
print(f"exactitud        : {len(t_exactitud)} filas")

tipo_transicion  : 5 filas
exactitud        : 13 filas


In [20]:
# ---- referencia (corrige la violacion de 1FN: listas separadas por comas) ----
def expandir_referencias(serie, tipo_fuente):
    filas = []
    for v in serie.dropna().astype(str):
        for token in v.split(","):
            token = token.strip()
            if token:
                filas.append((token, tipo_fuente))
    return filas

refs = (expandir_referencias(df_toclean["tp_ref"], "probabilidad")
        + expandir_referencias(df_toclean["line_ref"], "linea")
        + expandir_referencias(df_niv["Reference"], "nivel"))

t_referencia = (pd.DataFrame(refs, columns=["codigo", "tipo_fuente"])
                .drop_duplicates(subset="codigo", keep="first")
                .reset_index(drop=True))
print(f"referencia       : {len(t_referencia)} filas (catalogo de codigos bibliograficos unicos)")

referencia       : 377 filas (catalogo de codigos bibliograficos unicos)


In [21]:
# ---- espectro ----------------------------------------------------------------
espectros = sorted(df_toclean["espectro"].dropna().unique(),
                   key=lambda e: (NUM_ATOMICO[e.split()[0]], e.split()[1]))
t_espectro = pd.DataFrame({
    "id_espectro": range(1, len(espectros) + 1),
    "notacion": espectros,
})
t_espectro["id_elemento"] = t_espectro["notacion"].str.split().str[0].map(map_elemento)
t_espectro["estado_ionizacion"] = t_espectro["notacion"].str.split().str[1].map({"I": 1, "II": 2})
t_espectro = t_espectro[["id_espectro", "id_elemento", "estado_ionizacion", "notacion"]]
map_espectro = dict(zip(t_espectro["notacion"], t_espectro["id_espectro"]))
print(f"espectro         : {len(t_espectro)} filas")

espectro         : 18 filas


In [24]:
# ---- nivel_energia (incluye los niveles SIN linea asociada) ------------------
t_nivel_energia = df_niv.rename(columns={
    "Level ID": "id_nivel", "Configuration": "configuracion", "Term": "termino",
}).copy()
t_nivel_energia["id_espectro"] = t_nivel_energia["espectro"].map(map_espectro)
t_nivel_energia = t_nivel_energia[[
    "id_nivel", "id_espectro", "configuracion", "termino", "j_valor", "j_texto",
    "j_resuelto", "g", "energia_ev", "energia_interpolada", "incertidumbre_ev",
]].dropna(subset=["id_nivel"])

# g debe ser entero o nulo (MySQL SMALLINT UNSIGNED)
t_nivel_energia["g"] = t_nivel_energia["g"].astype("Int64")

assert t_nivel_energia["id_nivel"].is_unique, "id_nivel no es unico: violaria la PK"
print(f"nivel_energia    : {len(t_nivel_energia):,} filas "
      f"(PK 'id_nivel' verificada como unica)")

nivel_energia    : 5,752 filas (PK 'id_nivel' verificada como unica)


In [25]:
# ---- linea_espectral -----------------------------------------------------------
t_linea = df_toclean.copy()
t_linea["id_espectro"] = t_linea["espectro"].map(map_espectro)
t_linea["id_nivel_inferior"] = t_linea["ID_i"]
t_linea["id_nivel_superior"] = t_linea["ID_k"]
t_linea["codigo_tipo_transicion"] = t_linea["tipo_transicion"]
t_linea["codigo_exactitud"] = t_linea["exactitud"].astype(object).where(t_linea["exactitud"].notna(), None)
t_linea["longitud_onda_obs_nm_final"] = t_linea["longitud_onda_obs_nm"]

# id_nivel_* debe apuntar a un nivel que exista en t_nivel_energia; si no, se pone NULL
# (215 lineas del propio NIST no referencian un ID de nivel resoluble).
niveles_validos = set(t_nivel_energia["id_nivel"])
t_linea.loc[~t_linea["id_nivel_inferior"].isin(niveles_validos), "id_nivel_inferior"] = None
t_linea.loc[~t_linea["id_nivel_superior"].isin(niveles_validos), "id_nivel_superior"] = None

COLUMNAS_LINEA = {
    "id_espectro": "id_espectro",
    "id_nivel_inferior": "id_nivel_inferior",
    "id_nivel_superior": "id_nivel_superior",
    "codigo_tipo_transicion": "codigo_tipo_transicion",
    "codigo_exactitud": "codigo_exactitud",
    "longitud_onda_nm": "longitud_onda_nm",
    "longitud_onda_origen": "longitud_onda_origen",
    "medio": "medio",
    "longitud_onda_obs_nm_final": "longitud_onda_obs_nm",
    "longitud_onda_ritz_nm": "longitud_onda_ritz_nm",
    "ritz_es_limite": "ritz_es_limite",
    "incertidumbre_obs_nm": "incertidumbre_obs_nm",
    "incertidumbre_ritz_nm": "incertidumbre_ritz_nm",
    "aki_s1": "aki_s1",
    "fuerza_oscilador_fik": "fuerza_oscilador_fik",
    "log_gf": "log_gf",
    "intensidad_valor": "intensidad_valor",
    "intensidad_bandera": "intensidad_bandera",
    "energia_inferior_cm1": "energia_inferior_cm1",
    "energia_superior_cm1": "energia_superior_cm1",
    "energia_inferior_cm1_interpolada": "energia_inferior_interpolada",
    "energia_superior_cm1_interpolada": "energia_superior_interpolada",
    "energia_inferior_cm1_origen_desconocido": "energia_inferior_origen_desconocido",
    "energia_superior_cm1_origen_desconocido": "energia_superior_origen_desconocido",
    "medicion_repetida": "medicion_repetida",
}
t_linea_espectral = t_linea[list(COLUMNAS_LINEA.keys())].rename(columns=COLUMNAS_LINEA).copy()
t_linea_espectral.insert(0, "id_linea", range(1, len(t_linea_espectral) + 1))
t_linea_espectral["tp_ref"] = t_linea["tp_ref"].values
t_linea_espectral["line_ref"] = t_linea["line_ref"].values

assert t_linea_espectral["longitud_onda_nm"].notna().all(), "linea sin longitud de onda"
print(f"linea_espectral  : {len(t_linea_espectral):,} filas")

linea_espectral  : 18,288 filas


In [26]:
# ---- tablas puente: linea_referencia y nivel_referencia -----------------------
def construir_puente_lineas(df, col_id, col_ref, rol):
    filas = []
    sub = df[[col_id, col_ref]].dropna(subset=[col_ref])
    for id_linea, refs_txt in zip(sub[col_id], sub[col_ref].astype(str)):
        for token in refs_txt.split(","):
            token = token.strip()
            if token:
                filas.append((id_linea, token, rol))
    return pd.DataFrame(filas, columns=["id_linea", "codigo_referencia", "rol"])


t_linea_referencia = pd.concat([
    construir_puente_lineas(t_linea_espectral, "id_linea", "tp_ref", "probabilidad"),
    construir_puente_lineas(t_linea_espectral, "id_linea", "line_ref", "longitud_onda"),
], ignore_index=True).drop_duplicates()

# Solo referencias que existan en el catalogo (integridad referencial garantizada).
t_linea_referencia = t_linea_referencia[
    t_linea_referencia["codigo_referencia"].isin(t_referencia["codigo"])
].reset_index(drop=True)

t_linea_espectral = t_linea_espectral.drop(columns=["tp_ref", "line_ref"])

print(f"linea_referencia : {len(t_linea_referencia):,} filas")

linea_referencia : 24,757 filas


In [27]:
def construir_puente_niveles(df, col_id, col_ref):
    filas = []
    sub = df[[col_id, col_ref]].dropna(subset=[col_ref])
    for id_nivel, refs_txt in zip(sub[col_id], sub[col_ref].astype(str)):
        for token in refs_txt.split(","):
            token = token.strip()
            if token:
                filas.append((id_nivel, token))
    return pd.DataFrame(filas, columns=["id_nivel", "codigo_referencia"])


t_nivel_referencia = construir_puente_niveles(df_niv.rename(columns={"Level ID": "id_nivel"}),
                                              "id_nivel", "Reference").drop_duplicates()
t_nivel_referencia = t_nivel_referencia[
    t_nivel_referencia["codigo_referencia"].isin(t_referencia["codigo"])
    & t_nivel_referencia["id_nivel"].isin(t_nivel_energia["id_nivel"])
].reset_index(drop=True)

print(f"nivel_referencia : {len(t_nivel_referencia):,} filas")

nivel_referencia : 3,592 filas


### Verificación de integridad antes de cargar

Las mismas comprobaciones de la sección 18 de `03_limpieza_datos.ipynb`, aplicadas ahora
sobre los DataFrames que van a insertarse. Si alguna falla, no se llega a tocar MySQL.

In [28]:
TABLAS = {
    "elemento": t_elemento, "tipo_transicion": t_tipo_transicion, "exactitud": t_exactitud,
    "referencia": t_referencia, "espectro": t_espectro, "nivel_energia": t_nivel_energia,
    "linea_espectral": t_linea_espectral, "linea_referencia": t_linea_referencia,
    "nivel_referencia": t_nivel_referencia,
}

print("=== Resumen de las 9 tablas a cargar ===")
for nombre, df in TABLAS.items():
    print(f"  {nombre:<20} {len(df):>8,} filas x {df.shape[1]:>2} columnas")

# Unicidad de claves primarias
assert t_elemento["id_elemento"].is_unique and t_elemento["simbolo"].is_unique
assert t_espectro["id_espectro"].is_unique and t_espectro["notacion"].is_unique
assert t_nivel_energia["id_nivel"].is_unique
assert t_linea_espectral["id_linea"].is_unique
assert not t_linea_referencia.duplicated(subset=["id_linea", "codigo_referencia", "rol"]).any()
assert not t_nivel_referencia.duplicated(subset=["id_nivel", "codigo_referencia"]).any()

# Integridad referencial (0 huerfanas esperado)
huerfanas_esp = (~t_espectro["id_elemento"].isin(t_elemento["id_elemento"])).sum()
huerfanas_niv = (~t_nivel_energia["id_espectro"].isin(t_espectro["id_espectro"])).sum()
huerfanas_lin_esp = (~t_linea_espectral["id_espectro"].isin(t_espectro["id_espectro"])).sum()
huerfanas_lin_inf = (t_linea_espectral["id_nivel_inferior"].notna()
                     & ~t_linea_espectral["id_nivel_inferior"].isin(t_nivel_energia["id_nivel"])).sum()
huerfanas_lin_sup = (t_linea_espectral["id_nivel_superior"].notna()
                     & ~t_linea_espectral["id_nivel_superior"].isin(t_nivel_energia["id_nivel"])).sum()

for etiqueta, n in [("espectro->elemento", huerfanas_esp), ("nivel->espectro", huerfanas_niv),
                    ("linea->espectro", huerfanas_lin_esp), ("linea->nivel_inf", huerfanas_lin_inf),
                    ("linea->nivel_sup", huerfanas_lin_sup)]:
    assert n == 0, f"FK huerfana en {etiqueta}: {n} casos"
    print(f"  FK {etiqueta:<20} 0 huerfanas")

print("\nTODAS LAS VERIFICACIONES DE INTEGRIDAD PASARON. Se puede cargar en MySQL.")

=== Resumen de las 9 tablas a cargar ===
  elemento                   10 filas x  4 columnas
  tipo_transicion             5 filas x  4 columnas
  exactitud                  13 filas x  4 columnas
  referencia                377 filas x  2 columnas
  espectro                   18 filas x  4 columnas
  nivel_energia           5,752 filas x 11 columnas
  linea_espectral        18,288 filas x 26 columnas
  linea_referencia       24,757 filas x  3 columnas
  nivel_referencia        3,592 filas x  2 columnas
  FK espectro->elemento   0 huerfanas
  FK nivel->espectro      0 huerfanas
  FK linea->espectro      0 huerfanas
  FK linea->nivel_inf     0 huerfanas
  FK linea->nivel_sup     0 huerfanas

TODAS LAS VERIFICACIONES DE INTEGRIDAD PASARON. Se puede cargar en MySQL.


---
# Sección 6 — Carga en MySQL (vaciar y recargar)

Para que el notebook sea **repetible sin errores de clave duplicada**, cada ejecución vacía
las 9 tablas (en orden inverso de dependencias, con las validaciones de clave foránea
desactivadas temporalmente) y las vuelve a poblar desde cero, en **orden de dependencia**.
Todo ocurre dentro de una única transacción: si algo falla a mitad de camino, no queda la
base de datos a medias.

**No hay ninguna inserción manual**: todo el proceso es código ejecutable de punta a punta.

In [29]:
ORDEN_CARGA = [
    "elemento", "tipo_transicion", "exactitud", "referencia",
    "espectro", "nivel_energia", "linea_espectral",
    "linea_referencia", "nivel_referencia",
]

with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0"))
    for tabla in reversed(ORDEN_CARGA):
        conn.execute(text(f"TRUNCATE TABLE {tabla}"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1"))

print("Las 9 tablas fueron vaciadas. Listas para la carga.")

Las 9 tablas fueron vaciadas. Listas para la carga.


In [30]:
inicio = time.time()

with engine.begin() as conn:
    for tabla in ORDEN_CARGA:
        df = TABLAS[tabla]
        df.to_sql(tabla, con=conn, if_exists="append", index=False, method="multi", chunksize=2000)
        print(f"  Cargada '{tabla}': {len(df):,} filas")

print(f"\nCarga completa en {time.time() - inicio:.1f} segundos.")

  Cargada 'elemento': 10 filas
  Cargada 'tipo_transicion': 5 filas
  Cargada 'exactitud': 13 filas
  Cargada 'referencia': 377 filas
  Cargada 'espectro': 18 filas
  Cargada 'nivel_energia': 5,752 filas
  Cargada 'linea_espectral': 18,288 filas
  Cargada 'linea_referencia': 24,757 filas
  Cargada 'nivel_referencia': 3,592 filas

Carga completa en 12.6 segundos.


---
# Sección 7 — Verificación posterior a la carga

Se comprueba, **consultando la base de datos** (no los DataFrames en memoria), que todo
llegó completo y sin romper la integridad referencial.

In [31]:
print("=== Filas en MySQL vs. filas generadas en Python ===")
with engine.connect() as conn:
    for tabla in ORDEN_CARGA:
        n_bd = conn.execute(text(f"SELECT COUNT(*) FROM {tabla}")).scalar()
        n_py = len(TABLAS[tabla])
        marca = "OK" if n_bd == n_py else "DIFERENCIA"
        print(f"  [{marca:<10}] {tabla:<20} MySQL={n_bd:>8,}   Python={n_py:>8,}")
        assert n_bd == n_py, f"Discrepancia en {tabla}"

print("\nTodas las tablas coinciden exactamente con lo generado en Python.")

=== Filas en MySQL vs. filas generadas en Python ===
  [OK        ] elemento             MySQL=      10   Python=      10
  [OK        ] tipo_transicion      MySQL=       5   Python=       5
  [OK        ] exactitud            MySQL=      13   Python=      13
  [OK        ] referencia           MySQL=     377   Python=     377
  [OK        ] espectro             MySQL=      18   Python=      18
  [OK        ] nivel_energia        MySQL=   5,752   Python=   5,752
  [OK        ] linea_espectral      MySQL=  18,288   Python=  18,288
  [OK        ] linea_referencia     MySQL=  24,757   Python=  24,757
  [OK        ] nivel_referencia     MySQL=   3,592   Python=   3,592

Todas las tablas coinciden exactamente con lo generado en Python.


In [32]:
print("=== Busqueda de claves foraneas huerfanas directamente en MySQL ===")
CONSULTAS_HUERFANAS = {
    "espectro -> elemento":
        "SELECT COUNT(*) FROM espectro e LEFT JOIN elemento el ON e.id_elemento=el.id_elemento WHERE el.id_elemento IS NULL",
    "nivel_energia -> espectro":
        "SELECT COUNT(*) FROM nivel_energia n LEFT JOIN espectro e ON n.id_espectro=e.id_espectro WHERE e.id_espectro IS NULL",
    "linea_espectral -> nivel_inferior":
        "SELECT COUNT(*) FROM linea_espectral l LEFT JOIN nivel_energia n ON l.id_nivel_inferior=n.id_nivel "
        "WHERE l.id_nivel_inferior IS NOT NULL AND n.id_nivel IS NULL",
    "linea_espectral -> nivel_superior":
        "SELECT COUNT(*) FROM linea_espectral l LEFT JOIN nivel_energia n ON l.id_nivel_superior=n.id_nivel "
        "WHERE l.id_nivel_superior IS NOT NULL AND n.id_nivel IS NULL",
    "linea_referencia -> referencia":
        "SELECT COUNT(*) FROM linea_referencia lr LEFT JOIN referencia r ON lr.codigo_referencia=r.codigo "
        "WHERE r.codigo IS NULL",
}

with engine.connect() as conn:
    for etiqueta, sql in CONSULTAS_HUERFANAS.items():
        n = conn.execute(text(sql)).scalar()
        assert n == 0, f"{etiqueta}: {n} huerfanas"
        print(f"  [OK] {etiqueta:<35} 0 huerfanas")

print("\nIntegridad referencial verificada directamente en MySQL: 0 huerfanas.")

=== Busqueda de claves foraneas huerfanas directamente en MySQL ===
  [OK] espectro -> elemento                0 huerfanas
  [OK] nivel_energia -> espectro           0 huerfanas
  [OK] linea_espectral -> nivel_inferior   0 huerfanas
  [OK] linea_espectral -> nivel_superior   0 huerfanas
  [OK] linea_referencia -> referencia      0 huerfanas

Integridad referencial verificada directamente en MySQL: 0 huerfanas.


### La consulta que demuestra que el modelo funciona: doble `JOIN` a `nivel_energia`

Una línea espectral es un salto entre dos niveles. Esta consulta reconstruye la transición
completa —elemento, longitud de onda, `Aki` y la configuración electrónica de los dos
niveles— usando la clave foránea **dos veces sobre la misma tabla**, que es la particularidad
central de este modelo relacional.

In [33]:
CONSULTA_DOBLE_JOIN = '''
SELECT
    el.simbolo                    AS elemento,
    e.notacion                    AS espectro,
    l.longitud_onda_nm,
    l.aki_s1,
    tt.nombre                     AS tipo_transicion,
    ni.configuracion               AS config_nivel_inferior,
    ni.termino                     AS termino_inferior,
    ns.configuracion               AS config_nivel_superior,
    ns.termino                     AS termino_superior
FROM linea_espectral l
JOIN espectro        e  ON e.id_espectro = l.id_espectro
JOIN elemento         el ON el.id_elemento = e.id_elemento
JOIN tipo_transicion  tt ON tt.codigo = l.codigo_tipo_transicion
LEFT JOIN nivel_energia ni ON ni.id_nivel = l.id_nivel_inferior
LEFT JOIN nivel_energia ns ON ns.id_nivel = l.id_nivel_superior
WHERE l.aki_s1 IS NOT NULL
ORDER BY l.aki_s1 DESC
LIMIT 10;
'''

df_resultado = pd.read_sql(text(CONSULTA_DOBLE_JOIN), engine)
print("Las 10 lineas con Aki mas alto (transiciones mas rapidas), leidas de vuelta desde MySQL:")
display(df_resultado)

Las 10 lineas con Aki mas alto (transiciones mas rapidas), leidas de vuelta desde MySQL:


,elemento,espectro,longitud_onda_nm,aki_s1,tipo_transicion,config_nivel_inferior,termino_inferior,config_nivel_superior,termino_superior
0,C,C II,4.3169,7.000000e+11,Dipolo electrico,2s.2p2,4P,1s.2s.(3S). 2p3.(4S*),4S*
1,C,C II,4.2988,6.800000e+11,Dipolo electrico,2s2.2p,2P*,1s.2s2.2p2,2P
2,C,C II,4.2988,5.500000e+11,Dipolo electrico,2s2.2p,2P*,1s.2s2.2p2,2P
3,C,C II,4.3169,5.000000e+11,Dipolo electrico,2s.2p2,4P,1s.2s.(3S). 2p3.(4S*),4S*
4,C,C II,4.3060,3.000000e+11,Dipolo electrico,2s2.2p,2P*,1s.2s2.2p2,2D
5,C,C II,4.2988,2.700000e+11,Dipolo electrico,2s2.2p,2P*,1s.2s2.2p2,2P
6,C,C II,4.3150,2.500000e+11,Dipolo electrico,2s.2p2,4P,1s.2s.(3S). 2p3.(2D*),4D*
7,C,C II,4.3060,2.500000e+11,Dipolo electrico,2s2.2p,2P*,1s.2s2.2p2,2D
8,C,C II,4.3169,2.300000e+11,Dipolo electrico,2s.2p2,4P,1s.2s.(3S). 2p3.(4S*),4S*
9,C,C II,4.3150,2.100000e+11,Dipolo electrico,2s.2p2,4P,1s.2s.(3S). 2p3.(2D*),4D*


In [34]:
# Cierra el ciclo Python -> MySQL -> Python: la vista v_linea_analisis calcula
# la region espectral y log10(Aki) sin que esas columnas existan en la tabla.
CONSULTA_VISTA = '''
SELECT elemento, region_espectral, COUNT(*) AS num_lineas,
       ROUND(AVG(log10_aki), 2) AS log10_aki_promedio
FROM v_linea_analisis
WHERE log10_aki IS NOT NULL
GROUP BY elemento, region_espectral
ORDER BY elemento, region_espectral;
'''

df_vista = pd.read_sql(text(CONSULTA_VISTA), engine)
print(f"Filas devueltas por la vista v_linea_analisis: {len(df_vista)}")
display(df_vista.head(15))

Filas devueltas por la vista v_linea_analisis: 38


,elemento,region_espectral,num_lineas,log10_aki_promedio
0,B,Infrarrojo,264,4.41
1,B,Microondas/Radio,6,-11.55
2,B,Ultravioleta,320,6.48
3,B,Visible,114,5.86
4,Be,Infrarrojo,276,5.12
5,Be,Microondas/Radio,4,-14.59
6,Be,Ultravioleta,173,7.07
7,Be,Visible,90,5.14
8,C,Infrarrojo,1337,5.04
9,C,Rayos X,27,11.12


---
# Sección 8 — Exportación de subproductos

Se sobrescriben los CSV de `datos_procesados/` con el resultado de esta corrida (deben
coincidir con los de `03_limpieza_datos.ipynb`, porque aplican la misma lógica de limpieza),
como evidencia adicional de que el pipeline es reproducible de punta a punta.

In [35]:
t_linea_espectral.to_csv(os.path.join(DIR_PROCESADOS, "lineas_limpias.csv"), index=False)
t_nivel_energia.to_csv(os.path.join(DIR_PROCESADOS, "niveles_limpios.csv"), index=False)

for nombre, df in TABLAS.items():
    df.to_csv(os.path.join(DIR_PROCESADOS, f"tabla_{nombre}.csv"), index=False)

print("Subproductos exportados a datos_procesados/.")
print()
print("=" * 72)
print("PIPELINE COMPLETO: NIST ASD -> limpieza -> MySQL")
print("=" * 72)
print(f"Ejecutado el {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Filas totales cargadas en MySQL: {sum(len(df) for df in TABLAS.values()):,}")
print("Este notebook puede volver a ejecutarse en cualquier momento: vacia y")
print("recarga las 9 tablas sin intervencion manual.")

Subproductos exportados a datos_procesados/.

PIPELINE COMPLETO: NIST ASD -> limpieza -> MySQL
Ejecutado el 2026-09-21 06:37:19
Filas totales cargadas en MySQL: 52,812
Este notebook puede volver a ejecutarse en cualquier momento: vacia y
recarga las 9 tablas sin intervencion manual.
